# Phase 5 — model comparison

All candidates are fit on `train.csv` only. Validation is used for comparison; test is not touched here.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "processed" / "train.csv").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from ml.evaluation.pipeline import load_splits, build_model_specs, classification_metrics, select_threshold
splits = load_splits(ROOT)
X_train, y_train = splits["train"]
X_validation, y_validation = splits["validation"]
X_test, y_test = splits["test"]
models = build_model_specs(X_train, y_train)
fitted = {}
rows = []
for name, estimator in models.items():
    fitted[name] = estimator.fit(X_train, y_train)
    score = fitted[name].predict_proba(X_validation)[:, 1]
    rows.append({"model": name, **classification_metrics(y_validation, score)})
comparison = pd.DataFrame(rows).sort_values("roc_auc", ascending=False)
comparison